# Heart Disease Prediction using Random Forest Classifier

This notebook implements a heart disease prediction model using the Random Forest Classifier. It includes data loading, preprocessing, and model evaluation.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pickle
import warnings
warnings.filterwarnings('ignore')

## 1. Load Dataset

In [2]:
# ====================================================================
# 🌐 DATASET AUTO-RESOLVER & KAGGLE CONNECTOR (Heart Disease Dataset)
# ====================================================================
import os
import urllib.request
import pandas as pd

candidate_files = [
    '../datasets/heart.csv',
    './datasets/heart.csv',
    './heart.csv',
    '/kaggle/input/heart-disease-data/heart.csv',
    '/kaggle/input/heart-disease-uci/heart.csv'
]

csv_path = None
for f in candidate_files:
    if os.path.exists(f):
        csv_path = f
        print(f"✅ Found dataset file at: {csv_path}")
        break

if not csv_path:
    print("⚡ Dataset not found locally. Attempting KaggleHub download...")
    try:
        import kagglehub
        download_dir = kagglehub.dataset_download("johnsmith82/heart-disease-data")
        for root, _, files in os.walk(download_dir):
            for file in files:
                if file.endswith('.csv'):
                    csv_path = os.path.join(root, file)
                    print(f"✅ Downloaded via KaggleHub: {csv_path}")
                    break
    except Exception as e:
        print(f"⚠️ KaggleHub notice: {e}. Falling back to online mirror...")
        fallback_url = "https://raw.githubusercontent.com/amankharwal/Website-data/master/heart.csv"
        os.makedirs("datasets", exist_ok=True)
        csv_path = "datasets/heart.csv"
        urllib.request.urlretrieve(fallback_url, csv_path)
        print(f"✅ Downloaded mirror to: {csv_path}")

df = pd.read_csv(csv_path)
print(f"📊 Loaded Dataset Shape: {df.shape}")
df.head()


⚡ Dataset not found locally. Attempting KaggleHub download...
⚠️ KaggleHub notice: POST failed with: {"errors":["Not found"],"error":{"code":5},"wasSuccessful":false}. Falling back to online mirror...
✅ Downloaded mirror to: datasets/heart.csv
📊 Loaded Dataset Shape: (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


## 2. Preprocessing & Split
We will scale the features to ensure better model performance.

In [3]:
X = df.drop('target', axis=1)
y = df['target']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

## 3. Model Training (Random Forest)

In [4]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

## 4. Evaluation

In [5]:
y_train_pred = rf.predict(X_train)
y_test_pred = rf.predict(X_test)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

Training Accuracy: 1.0000
Test Accuracy: 0.8361

Confusion Matrix:
[[24  5]
 [ 5 27]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.83      0.83        29
           1       0.84      0.84      0.84        32

    accuracy                           0.84        61
   macro avg       0.84      0.84      0.84        61
weighted avg       0.84      0.84      0.84        61



## 5. Save Model

In [6]:
# ====================================================================
# 💾 SAVE TRAINED MODEL & ARTIFACTS
# ====================================================================
import os
import pickle

os.makedirs('../models/heart_model', exist_ok=True)
os.makedirs('../models', exist_ok=True)

pickle.dump(rf, open('heart_model.pkl', 'wb'))
pickle.dump(scaler, open('heart_scaler.pkl', 'wb'))

try:
    pickle.dump(rf, open('../models/heart_model/heart_model.pkl', 'wb'))
    pickle.dump(scaler, open('../models/heart_model/heart_scaler.pkl', 'wb'))
    pickle.dump(rf, open('../models/heart_model.pkl', 'wb'))
    pickle.dump(scaler, open('../models/heart_scaler.pkl', 'wb'))
except Exception as e:
    pass

print('✅ Heart Disease Model saved as heart_model.pkl and heart_scaler.pkl')


✅ Heart Disease Model saved as heart_model.pkl and heart_scaler.pkl


## 6. Verification
Load the saved model and test on a random sample from the test set.

In [7]:
# Verification
import pickle

# Load model
model = pickle.load(open('heart_model.pkl', 'rb'))

# Test on a sample from X_test (assuming X_test is in memory from execution)
try:
    sample_idx = 0
    sample_data = X_test[sample_idx].reshape(1, -1)
    prediction = model.predict(sample_data)

    print(f"Predicted Class: {prediction[0]}")
    # Handle different y_test types
    actual = y_test.iloc[sample_idx] if hasattr(y_test, 'iloc') else y_test[sample_idx] if isinstance(y_test, (pd.Series, np.ndarray)) else y_test
    print(f"Actual Class: {actual}")
except NameError:
    print("X_test not found. Please run the notebook cells above to load data first.")

Predicted Class: 0
Actual Class: 0


## 📥 Download Trained Model & Artifacts (Kaggle)
This cell creates a 1-click downloadable ZIP archive containing the trained model and scaler.

In [8]:
# ====================================================================
# 📥 SAVE & DOWNLOAD TRAINED ARTIFACTS ON KAGGLE
# ====================================================================
import os
import zipfile
from IPython.display import FileLink, display

saved_files = ['heart_model.pkl', 'heart_scaler.pkl']
saved_files = [f for f in saved_files if os.path.exists(f)]

zip_filename = 'heart_artifacts.zip'
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file in saved_files:
        zipf.write(file)
        print(f'📦 Packaged in zip: {file}')

print('')
print('' + f'✅ Bundle ready: {zip_filename} ({os.path.getsize(zip_filename)} bytes)')
print('👉 Click the link below to download your trained artifacts:')
display(FileLink(zip_filename))


📦 Packaged in zip: heart_model.pkl
📦 Packaged in zip: heart_scaler.pkl

✅ Bundle ready: heart_artifacts.zip (114125 bytes)
👉 Click the link below to download your trained artifacts:


/kaggle/working/heart_artifacts.zip